# WeavEHR ↔ RICU validation for all four ICU datasets

This notebook validates the 48 YAIB/RICU dynamic variables for the four datasets currently produced by the WeavEHR example pipeline:

- `mimic-iv` 3.1
- `eicu-crd` 2.0
- `hirid` 1.1.1
- `aumc` 1.5.0

The comparison is bounded to the first 7 days (`168 h`) and uses `aggregation_mode="ricu"`.

**Important:** the repository exposes several diagnostics (`coverage`, `missingness`, `value_diff`, `reproduction_accuracy`, …). It does not define one canonical `<1%` acceptance metric, so this notebook deliberately does not invent a pass/fail rule. It generates and displays all relevant reports so the agreed per-concept metric can be applied afterwards.

AUMC is a special case: WeavEHR uses OMOP `visit_occurrence_id`, while RICU uses native AmsterdamUMCdb admission IDs. Without an explicit stay crosswalk, the comparison may correctly report that direct stay-level scoring is not comparable.

In [4]:
import os
import subprocess
from pathlib import Path

import polars as pl

from weavehr_yaib import (
    build_and_write_yaib_wide_for_dataset,
    compare_weavehr_wide_to_ricu_for_dataset,
    concept_root_from_output,
    resolve_weavehr_workspace,
)

MAX_HOURS = 7 * 24

WEAVEHR_OUTPUT = Path(
    "/home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project"
).resolve()

REPO_ROOT = Path("/home/q039tl/workspace/weavehr-yaib").resolve()
RICU_CONCEPT_DICT = (
    Path.home()
    / "workspace"
    / "ricu"
    / "inst"
    / "extdata"
    / "config"
    / "concept-dict.json"
)

WORKSPACE = resolve_weavehr_workspace(WEAVEHR_OUTPUT)
CONCEPT_ROOT = concept_root_from_output(WEAVEHR_OUTPUT)

DATASETS = {
    "mimic-iv": {
        "version": "3.1",
        "ricu_source": "miiv",
        "ricu_script": "scripts/datasets/export_ricu_mimic-iv.R",
    },
    "eicu-crd": {
        "version": "2.0",
        "ricu_source": "eicu",
        "ricu_script": "scripts/datasets/export_ricu_eicu-crd.R",
    },
    "hirid": {
        "version": "1.1.1",
        "ricu_source": "hirid",
        "ricu_script": "scripts/datasets/export_ricu_hirid.R",
    },
    "aumc": {
        "version": "1.5.0",
        "ricu_source": "aumc",
        "ricu_script": "scripts/datasets/export_ricu_aumc.R",
    },
}

print("WeavEHR project:", WEAVEHR_OUTPUT)
print("WeavEHR workspace:", WORKSPACE)
print("Concept root:", CONCEPT_ROOT)
print("weavehr-yaib repo:", REPO_ROOT)
print("RICU concept dict:", RICU_CONCEPT_DICT)


WeavEHR project: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project
WeavEHR workspace: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace
Concept root: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/concept
weavehr-yaib repo: /home/q039tl/workspace/weavehr-yaib
RICU concept dict: /home/q039tl/workspace/ricu/inst/extdata/config/concept-dict.json


## Preconditions

The RICU export uses the repository's dataset-specific R scripts. `RICU_DATA_PATH` must already point to the local RICU source data.

Optional per-dataset source-version declarations can be set before running Jupyter, for example:

```bash
export RICU_SOURCE_VERSION_MIMIC_IV=2.2
export RICU_SOURCE_VERSION_EICU_CRD=2.0
export RICU_SOURCE_VERSION_HIRID=1.1.1
export RICU_SOURCE_VERSION_AUMC=1.0.2
```

Only set these when they match the data actually loaded by RICU. If unset, the comparison can still run, but provenance may be reported as assumed/unverified.

In [5]:
%env RICU_DATA_PATH=/home/q039tl/ricu_data

env: RICU_DATA_PATH=/home/q039tl/ricu_data


In [6]:
required_paths = {
    "WEAVEHR_OUTPUT": WEAVEHR_OUTPUT,
    "REPO_ROOT": REPO_ROOT,
    "RICU_CONCEPT_DICT": RICU_CONCEPT_DICT,
}

for name, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

if "RICU_DATA_PATH" not in os.environ:
    raise ValueError(
        "Set RICU_DATA_PATH in the environment before running the RICU export cells."
    )

print("Preconditions OK")


Preconditions OK


## 1. Export the RICU references

Each dataset gets its own output directory below:

`<WeavEHR workspace>/yaib/<dataset>/`

The dataset-specific wrapper scripts select the corresponding RICU source.

In [ ]:
for dataset, cfg in DATASETS.items():
    dataset_output = WORKSPACE / "yaib" / dataset
    dataset_output.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env["RICU_OUT_DIR"] = str(dataset_output)
    env["RICU_SRC_LOAD"] = "mimic,mimic_demo,eicu,eicu_demo,hirid,aumc,miiv,sic"
    env.pop("RICU_CONFIG_PATH", None)
    env["R_ENVIRON_USER"] = "/dev/null"

    version_env = "RICU_SOURCE_VERSION_" + dataset.upper().replace("-", "_")
    declared_reference_version = os.environ.get(version_env)
    if declared_reference_version:
        env["RICU_SOURCE_VERSION"] = declared_reference_version
    else:
        env.pop("RICU_SOURCE_VERSION", None)

    print(f"\n=== RICU export: {dataset} ===")
    print("output:", dataset_output)
    print(
        "declared reference version:",
        declared_reference_version if declared_reference_version else "<not declared>",
    )

    subprocess.run(
        [
            "Rscript",
            "--no-environ",
            cfg["ricu_script"],
        ],
        cwd=REPO_ROOT,
        check=True,
        env=env,
    )


## 2. Build WeavEHR 7-day YAIB-wide tables and compare to RICU

This uses the same repository API as the existing per-dataset notebooks and explicitly selects the WeavEHR dataset versions from the current pipeline run.

In [7]:
wide_results = {}
comparisons = {}

for dataset, cfg in DATASETS.items():
    print(f"\n{'=' * 80}")
    print(f"{dataset} {cfg['version']}")
    print("=" * 80)

    dataset_output = WORKSPACE / "yaib" / dataset

    weavehr_7d = build_and_write_yaib_wide_for_dataset(
        dataset=dataset,
        dataset_version=cfg["version"],
        max_hours=MAX_HOURS,
        output_root=dataset_output,
        concept_root=CONCEPT_ROOT,
        icustays_csv=None,
        ricu_concept_dict=RICU_CONCEPT_DICT,
        version="1.0.0",
        aggregation_mode="ricu",
    )

    compare_kwargs = dict(
        dataset=dataset,
        dataset_version=cfg["version"],
        max_hours=MAX_HOURS,
        output_root=dataset_output,
        concept_root=CONCEPT_ROOT,
        icustays_csv=None,
        ricu_concept_dict=RICU_CONCEPT_DICT,
        weavehr_wide_path=weavehr_7d.output_path,
    )

    # AUMC uses different stay-ID spaces on the two sides.
    # Without a verified crosswalk, leave it explicitly unset.
    if dataset == "aumc":
        compare_kwargs["stay_crosswalk"] = None

    comparison = compare_weavehr_wide_to_ricu_for_dataset(**compare_kwargs)

    wide_results[dataset] = weavehr_7d
    comparisons[dataset] = comparison

    print("WeavEHR wide:", weavehr_7d.output_path)
    print("reports:", comparison.reports_dir)



mimic-iv 3.1
WeavEHR wide: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/mimic-iv/weavehr_dyn_168h.parquet
reports: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/mimic-iv/reports/168h/weavehr-3.1__ricu-miiv-2.2

eicu-crd 2.0
WeavEHR wide: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/eicu-crd/weavehr_dyn_168h.parquet
reports: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/eicu-crd/reports/168h/weavehr-2.0__ricu-eicu-2.0

hirid 1.1.1
WeavEHR wide: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/hirid/weavehr_dyn_168h.parquet
reports: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/hirid/reports/168h/weavehr-1.1.1__ricu-hirid-1.1.1

aumc 1.5.0
WeavEHR wide: /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/

## 3. Provenance and comparison scope

Check this first. In particular:

- MIMIC-IV may be a cross-version comparison if RICU uses 2.2 while WeavEHR uses 3.1.
- eICU and HiRID should normally be same-version when the corresponding RICU data versions match.
- AUMC may be `not_comparable` at stay level without a verified native↔OMOP crosswalk.

In [8]:
provenance = pl.concat(
    [
        result.provenance.with_columns(pl.lit(dataset).alias("dataset"))
        for dataset, result in comparisons.items()
    ],
    how="diagonal_relaxed",
)

provenance


weavehr_dataset,weavehr_dataset_version,weavehr_version_source,weavehr_version_verified,weavehr_representation,reference_dataset,reference_dataset_version,reference_version_source,reference_version_verified,reference_representation,same_version,versions_verified,comparison_scope,validation_level,metrics_restricted_to_shared_scope,weavehr_stay_id_space,reference_stay_id_space,stay_comparison_basis,stay_comparison_confidence,stay_id_version_assumption,n_scope_shared_stays,n_scope_weavehr_only_stays,n_scope_reference_only_stays,n_scope_reference_stays_without_crosswalk,n_scope_shared_concepts,n_scope_weavehr_only_concepts,n_scope_reference_only_concepts,stay_relationship,concept_relationship,max_hours,weavehr_path,reference_path,dataset
str,str,str,bool,str,str,str,str,bool,str,bool,bool,str,str,bool,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,str,str,i64,str,str,str
"""mimic-iv""","""3.1""","""explicit""",true,"""native source tables (WeavEHR)""","""ricu:miiv""","""2.2""","""ricu_config""",false,"""native source tables (ricu)""",false,false,"""shared_subset""","""shared_subset_unverified_versi…",true,"""mimic-iv:icustays.stay_id""","""mimic-iv:icustays.stay_id""","""direct_identifier_match""","""medium""","""unverified_cross_version_stabi…",94444,14,0,0,48,0,0,"""weavehr_superset""","""equal""",168,"""/home/q039tl/output/WeavEHR.ex…","""/home/q039tl/output/WeavEHR.ex…","""mimic-iv"""
"""eicu-crd""","""2.0""","""explicit""",true,"""native source tables (WeavEHR)""","""ricu:eicu""","""2.0""","""ricu_config""",false,"""native source tables (ricu)""",true,false,"""shared_subset""","""shared_subset_unverified_versi…",true,"""eicu-crd:patient.patientunitst…","""eicu-crd:patient.patientunitst…","""direct_identifier_match""","""medium""","""unverified_cross_version_stabi…",200859,0,0,0,48,0,0,"""equal""","""equal""",168,"""/home/q039tl/output/WeavEHR.ex…","""/home/q039tl/output/WeavEHR.ex…","""eicu-crd"""
"""hirid""","""1.1.1""","""explicit""",true,"""native source tables (WeavEHR)""","""ricu:hirid""","""1.1.1""","""ricu_config""",false,"""native source tables (ricu)""",true,false,"""shared_subset""","""shared_subset_unverified_versi…",true,"""hirid:general.patientid""","""hirid:general.patientid""","""direct_identifier_match""","""medium""","""unverified_cross_version_stabi…",33904,1,0,0,48,0,0,"""weavehr_superset""","""equal""",168,"""/home/q039tl/output/WeavEHR.ex…","""/home/q039tl/output/WeavEHR.ex…","""hirid"""
"""aumc""","""1.5.0""","""explicit""",true,"""OMOP CDM 5.4 (AMSTEL ETL)""","""ricu:aumc""","""1.0.2""","""manual_default""",false,"""native source tables (ricu)""",false,false,"""shared_subset""","""not_comparable""",true,"""omop:visit_occurrence.visit_oc…","""aumcdb:admissions.admissionid""","""side_by_side_only""","""not_comparable""","""not_applicable""",null,null,null,0,48,0,0,"""not_directly_comparable""","""equal""",168,"""/home/q039tl/output/WeavEHR.ex…","""/home/q039tl/output/WeavEHR.ex…","""aumc"""


## 4. Combined report tables

These are the repository's existing diagnostics. No additional acceptance metric is introduced here.

In [9]:
def combine_report(attribute: str) -> pl.DataFrame:
    frames = []
    for dataset, result in comparisons.items():
        frame = getattr(result, attribute)
        if frame is None or frame.is_empty():
            continue
        frames.append(frame.with_columns(pl.lit(dataset).alias("dataset")))

    if not frames:
        return pl.DataFrame()

    return pl.concat(frames, how="diagonal_relaxed")


coverage_all = combine_report("coverage")
missingness_all = combine_report("missingness")
value_diff_all = combine_report("value_diff")
reproduction_accuracy_all = combine_report("reproduction_accuracy")
key_overlap_all = combine_report("key_overlap")
stay_overlap_all = combine_report("stay_overlap")

print("coverage")
display(coverage_all)

print("missingness")
display(missingness_all)

print("value_diff")
display(value_diff_all)

print("reproduction_accuracy")
display(reproduction_accuracy_all)


coverage


concept,weavehr_non_null,reference_non_null,diff_non_null,dataset
str,i64,i64,i64,str
"""alb""",63213,62813,400,"""mimic-iv"""
"""alp""",124079,123449,630,"""mimic-iv"""
"""alt""",124916,124258,658,"""mimic-iv"""
"""ast""",126508,125847,661,"""mimic-iv"""
"""be""",382320,380918,1402,"""mimic-iv"""
…,…,…,…,…
"""sbp""",1408378,1408378,0,"""hirid"""
"""temp""",369195,369195,0,"""hirid"""
"""tnt""",23580,23580,0,"""hirid"""


missingness


concept,both_non_null,only_weavehr,only_reference,both_null,dataset
str,u32,u32,u32,u32,str
"""alb""",62790,419,23,5932537,"""mimic-iv"""
"""alp""",123401,674,48,5871646,"""mimic-iv"""
"""alt""",124209,702,49,5870809,"""mimic-iv"""
"""ast""",125799,704,48,5869218,"""mimic-iv"""
"""be""",380896,1380,22,5613471,"""mimic-iv"""
…,…,…,…,…,…
"""sbp""",1408378,0,0,104665,"""hirid"""
"""temp""",369195,0,0,1143848,"""hirid"""
"""tnt""",23580,0,0,1489463,"""hirid"""


value_diff


concept,n_both_non_null,mean_abs_diff,median_abs_diff,max_abs_diff,mean_rel_diff,dataset
str,u32,f32,f32,f32,f32,str
"""alb""",62790,2.6580e-11,0.0,2.3842e-7,1.0504e-11,"""mimic-iv"""
"""alp""",123401,0.0,0.0,0.0,0.0,"""mimic-iv"""
"""alt""",124209,0.0,0.0,0.0,0.0,"""mimic-iv"""
"""ast""",125799,0.0,0.0,0.0,0.0,"""mimic-iv"""
"""be""",380896,0.0,0.0,0.0,0.0,"""mimic-iv"""
…,…,…,…,…,…,…
"""sbp""",1408378,5.4171e-11,0.0,0.000008,7.0525e-13,"""hirid"""
"""temp""",369195,6.6334e-9,0.0,0.000004,1.7904e-10,"""hirid"""
"""tnt""",23580,5.2436e-8,3.7253e-9,0.000061,4.5846e-8,"""hirid"""


reproduction_accuracy


weavehr_dataset_version,reference_dataset_version,same_version,versions_verified,comparison_scope,validation_level,metrics_restricted_to_shared_scope,weavehr_stay_id_space,reference_stay_id_space,stay_comparison_basis,stay_comparison_confidence,stay_id_version_assumption,n_scope_shared_stays,n_scope_weavehr_only_stays,n_scope_reference_only_stays,n_scope_reference_stays_without_crosswalk,n_scope_shared_concepts,n_scope_weavehr_only_concepts,n_scope_reference_only_concepts,n_stays_weavehr,n_stays_reference,n_stays_common,n_stays_only_weavehr,n_stays_only_reference,n_common_stays_identical,n_common_stays_not_identical,common_stay_exact_match_rate,stay_count_error,stay_count_absolute_error,mean_common_stay_row_count_error,mean_common_stay_absolute_row_count_error,summed_common_stay_row_count_error,summed_common_stay_absolute_row_count_error,total_row_count_error,total_absolute_row_count_error,n_rows_weavehr,n_rows_reference,n_rows_common_keys,n_rows_only_weavehr,n_rows_only_reference,n_rows_equal_content,n_rows_value_mismatch,row_content_disagreement_rate,dataset
str,str,bool,bool,str,str,bool,str,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,str
"""3.1""","""2.2""",false,false,"""shared_subset""","""shared_subset_unverified_versi…",true,"""mimic-iv:icustays.stay_id""","""mimic-iv:icustays.stay_id""","""direct_identifier_match""","""medium""","""unverified_cross_version_stabi…",94444,14,0,0,48,0,0,94444,94434,94434,10,0,279,94155,0.002954,0.000106,0.000106,0.090866,0.090866,0.045263,0.045263,0.045265,0.045265,6267170,5995769,5995769,271401,0,5156881,838888,0.17716,"""mimic-iv"""
"""2.0""","""2.0""",true,false,"""shared_subset""","""shared_subset_unverified_versi…",true,"""eicu-crd:patient.patientunitst…","""eicu-crd:patient.patientunitst…","""direct_identifier_match""","""medium""","""unverified_cross_version_stabi…",200859,0,0,0,48,0,0,200859,196340,196340,4519,0,72232,124108,0.367892,0.023016,0.023016,0.180292,0.180292,0.033528,0.033528,0.035365,0.035365,10808046,10438879,10438879,369167,0,10363794,75085,0.041104,"""eicu-crd"""
"""1.1.1""","""1.1.1""",true,false,"""shared_subset""","""shared_subset_unverified_versi…",true,"""hirid:general.patientid""","""hirid:general.patientid""","""direct_identifier_match""","""medium""","""unverified_cross_version_stabi…",33904,1,0,0,48,0,0,33904,33896,33896,8,0,176,33720,0.005192,0.000236,0.000236,0.088904,0.088904,0.04579,0.04579,0.045795,0.045795,1582333,1513043,1513043,69290,0,820704,692339,0.481333,"""hirid"""
"""1.5.0""","""1.0.2""",false,false,"""shared_subset""","""not_comparable""",true,"""omop:visit_occurrence.visit_oc…","""aumcdb:admissions.admissionid""","""side_by_side_only""","""not_comparable""","""not_applicable""",null,null,null,0,48,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""aumc"""


## 5. HiRID focus: the two newly implemented concepts

The YAIB/RICU names are:

- `lymph` → WeavEHR `lymphocytes`
- `urine` → WeavEHR `urine_output`

This cell extracts whichever comparison rows mention these two RICU variables.

In [10]:
def filter_concepts(df: pl.DataFrame, concepts: list[str]) -> pl.DataFrame:
    if df.is_empty():
        return df

    candidate_columns = [
        c
        for c in ("concept", "variable", "dynamic_var", "var", "feature")
        if c in df.columns
    ]

    if not candidate_columns:
        print("No known concept-name column found. Columns:", df.columns)
        return df

    col = candidate_columns[0]
    return df.filter(pl.col(col).is_in(concepts))


for report_name in ("coverage", "missingness", "value_diff", "per_stay_reproduction"):
    report = getattr(comparisons["hirid"], report_name)
    print(f"\n=== HiRID {report_name}: lymph + urine ===")
    display(filter_concepts(report, ["lymph", "urine"]))



=== HiRID coverage: lymph + urine ===


concept,weavehr_non_null,reference_non_null,diff_non_null
str,i64,i64,i64
"""lymph""",21996,21997,-1
"""urine""",831766,831766,0



=== HiRID missingness: lymph + urine ===


concept,both_non_null,only_weavehr,only_reference,both_null
str,u32,u32,u32,u32
"""lymph""",21996,0,1,1491046
"""urine""",831766,0,0,681277



=== HiRID value_diff: lymph + urine ===


concept,n_both_non_null,mean_abs_diff,median_abs_diff,max_abs_diff,mean_rel_diff
str,u32,f32,f32,f32,f32
"""lymph""",21996,0.000007,0.0,0.054862,0.000046
"""urine""",831766,0.012055,0.000015,2000.000122,0.00006



=== HiRID per_stay_reproduction: lymph + urine ===
No known concept-name column found. Columns: ['stay_id', 'n_rows_weavehr', 'n_rows_reference', 'n_only_weavehr_rows', 'n_only_reference_rows', 'n_common_rows', 'n_equal_rows', 'n_value_mismatch_rows', 'in_both', 'row_count_diff', 'row_count_relative_error', 'row_count_absolute_relative_error', 'content_identical']


stay_id,n_rows_weavehr,n_rows_reference,n_only_weavehr_rows,n_only_reference_rows,n_common_rows,n_equal_rows,n_value_mismatch_rows,in_both,row_count_diff,row_count_relative_error,row_count_absolute_relative_error,content_identical
i64,u32,u32,u32,u32,u32,u32,u32,bool,u32,f64,f64,bool
1,16,16,0,0,16,6,10,true,0,0.0,0.0,false
2,169,168,1,0,168,99,69,true,1,0.005952,0.005952,false
3,19,18,1,0,18,5,13,true,1,0.055556,0.055556,false
4,94,94,0,0,94,58,36,true,0,0.0,0.0,false
5,70,70,0,0,70,48,22,true,0,0.0,0.0,false
…,…,…,…,…,…,…,…,…,…,…,…,…
33901,159,157,2,0,157,86,71,true,2,0.012739,0.012739,false
33902,19,17,2,0,17,11,6,true,2,0.117647,0.117647,false
33903,23,23,0,0,23,15,8,true,0,0.0,0.0,false


## 6. Report locations

The comparison function writes the standard CSV reports itself, including:

- `provenance.csv`
- `scope_overlap.csv`
- `scope_differences.csv`
- `table_summary.csv`
- `stay_overlap.csv`
- `key_overlap.csv`
- `window_summary.csv`
- `window_differences.csv`
- `coverage.csv`
- `missingness.csv`
- `value_diff.csv`
- `per_stay_reproduction.csv`
- `reproduction_accuracy.csv`

Use these files for the subsequent debugging/documentation of concepts above the agreed technical threshold.

In [11]:
for dataset, result in comparisons.items():
    print(f"{dataset:10s} -> {result.reports_dir}")


mimic-iv   -> /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/mimic-iv/reports/168h/weavehr-3.1__ricu-miiv-2.2
eicu-crd   -> /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/eicu-crd/reports/168h/weavehr-2.0__ricu-eicu-2.0
hirid      -> /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/hirid/reports/168h/weavehr-1.1.1__ricu-hirid-1.1.1
aumc       -> /home/q039tl/output/WeavEHR.example/pipeline-weavehr-yaib/output/project/workspace/yaib/aumc/reports/168h/weavehr-1.5.0__ricu-aumc-1.0.2
